# Cancelamento de reservas de hotel: análise exploratória

**Pergunta:** quais reservas têm mais chance de ser canceladas, no momento em que entram?

**Para quê:** decidir em quais reservas pedir depósito ou ligar para confirmar, e quantos cancelamentos esperar para calibrar o overbooking.

**Este notebook** decide quais variáveis entram no modelo e de que forma. O modelo e a avaliação ficam em `02_modelo.ipynb`.

## Resumo

*(preenchido ao final da análise)*

- Taxa de cancelamento: __.
- Variáveis excluídas por vazamento: __.
- Escopo do modelo: __.
- Variáveis que entram, e de que forma: __.

## Contexto e abordagem

**Contexto.** Dois hotéis em Portugal, um urbano (City Hotel, em Lisboa) e um resort (Resort Hotel, no Algarve), com chegadas de julho de 2015 a agosto de 2017. Cada linha é uma reserva. Uma reserva cancelada (ou um no-show) pode virar quarto vazio: a diária não entra e muitas vezes não dá tempo de revender o quarto. Fonte: Antonio, Almeida e Nunes (2019), *Hotel booking demand datasets*, Data in Brief.

**Abordagem.** Regressão logística: o resultado é sim ou não (cancelou ou veio), e o negócio precisa de uma probabilidade de cancelamento por reserva para decidir onde agir.

**Premissa que guia a análise.** O modelo será usado **no momento em que a reserva entra**. Então só pode usar o que o hotel sabe nessa hora. Uma variável que só existe, ou só tem aquele valor, depois do check-in ou do cancelamento é **vazamento**: deixa o modelo ótimo no papel e inútil na prática.

**Validação.** Pela mesma premissa, o modelo é avaliado em reservas futuras: treino com as chegadas de 2015 e 2016, teste com as de 2017.

In [ ]:
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

sys.path.append('..')   # o módulo de funções do projeto fica em analise/
from funcoes_logistica import aplicar_estilo, resumo_colunas

aplicar_estilo()
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 80)

## 1. A base

In [ ]:
hotel = pd.read_csv('../../dados/hotel_bookings.csv')
print('Linhas e colunas:', hotel.shape)
hotel.head()

In [ ]:
resumo_colunas(hotel)

**Leitura.**

- **119.390 reservas e 32 colunas.** O `y` (`is_canceled`) já vem como 0/1: 44.224 canceladas (37,0%), contando os no-shows.
- **O tipo no Python nem sempre é o tipo da variável:**
  - `is_repeated_guest` é número, mas binária;
  - `agent` e `company` são decimais só porque têm missing: são códigos, ou seja, categóricas;
  - ano, semana e dia da chegada são números, mas são calendário.
- **Missing:** `company` (94,3%), `agent` (13,7%), `country` (488 reservas) e `children` (4 reservas).
- **Primeiros sinais para investigar:**
  - `reservation_status`: 43.017 Canceled + 1.207 No-Show = 44.224, exatamente as reservas canceladas. É o próprio `y` escrito de outro jeito: se entrasse no modelo, ele acertaria tudo no papel e não serviria para nada na prática. Sai (confirmar na seção de vazamento).
  - `deposit_type`: 88% das reservas são sem depósito (104.641), 12% Non Refund (14.587) e só 162 Refundable. A ação em discussão (pedir depósito ou ligar para confirmar) só faz sentido para quem ainda não deu garantia: candidato a definir o **escopo** do modelo, e não a entrar como variável. As 162 Refundable são poucas demais para estimar qualquer efeito.
  - `babies`: 0 em 99% das reservas (118.473). Uma variável quase constante separa pouco quem cancela de quem vem. Candidata a sair, ou a ser juntada com `children` ("tem criança ou bebê").
  - `distribution_channel`: 82% das reservas chegam por agências de viagem ou operadoras de turismo (TA/TO, *travel agents / tour operators*). O `market_segment` já separa esse grupo em agência online (Online TA) e agência ou operadora tradicional (Offline TA/TO), então o canal provavelmente repete o segmento com menos detalhe. Candidata a sair por redundância (confirmar com o V de Cramér entre os dois).

## 2. Dicionário de dados, primeira versão

O que cada variável é e o que esperamos dela **antes de olhar a relação com o cancelamento**. A coluna **Quando se sabe** aplica a premissa do momento da decisão:

- **na reserva:** o hotel tem a informação quando a reserva entra;
- **pode mudar até a chegada:** existe na reserva, mas a base guarda o valor do fim;
- **só no fim:** só existe depois do check-in ou do cancelamento (vazamento);
- **?:** não sabemos, e os dados vão mostrar.

| Variável | Descrição | Tipo | Quando se sabe | Pergunta a responder |
| :--- | :--- | :--- | :--- | :--- |
| **Hotel e antecedência** | | | | |
| `hotel` | City Hotel (Lisboa) ou Resort Hotel (Algarve) | categórica (2) | na reserva | O urbano cancela mais que o resort? |
| `lead_time` | Dias entre a reserva e a chegada | numérica | na reserva | Mais antecedência, mais chance de mudar de plano? A relação com o logito é linear? |
| **Data da chegada** | | | | |
| `arrival_date_year` | Ano da chegada | calendário (3) | na reserva | Não serve como variável: reserva futura sempre tem ano novo. Separa treino e teste. |
| `arrival_date_month` | Mês da chegada | categórica (12) | na reserva | Há sazonalidade? Dá para agrupar em temporadas? |
| `arrival_date_week_number` | Semana do ano da chegada | calendário (53) | na reserva | Acrescenta algo ao mês? |
| `arrival_date_day_of_month` | Dia do mês da chegada | calendário (31) | na reserva | Tem alguma lógica de negócio? |
| **Estadia e hóspedes** | | | | |
| `stays_in_weekend_nights` | Noites de sábado ou domingo | contagem | pode mudar até a chegada | Lazer (fim de semana) × trabalho (semana)? |
| `stays_in_week_nights` | Noites de segunda a sexta | contagem | pode mudar até a chegada | Vale somar com as noites de fim de semana? |
| `adults` | Adultos | contagem | na reserva | Reserva sem adulto é erro de cadastro? |
| `children` | Crianças | contagem | na reserva | Família cancela mais? |
| `babies` | Bebês | contagem | na reserva | Quase sempre zero: ajuda em algo? |
| `meal` | Regime: BB, HB, FB, SC, Undefined | categórica (5) | na reserva | Pela documentação, Undefined e SC são a mesma coisa (sem refeição). Os dados confirmam? |
| **Origem e canal** | | | | |
| `country` | País do hóspede | categórica (177) | ? | Pode ser preenchido só no check-in. Se for, é vazamento. |
| `market_segment` | Segmento: Online TA (agências online, como Booking.com), Offline TA/TO (agências tradicionais e operadoras, que costumam reservar blocos de quartos), Groups, Direct, Corporate... | categórica (8) | na reserva | Grupos e operadoras se comportam diferente? Algum nível tem poucas reservas? |
| `distribution_channel` | Canal: TA/TO (agências de viagem e operadoras de turismo), Direct, Corporate, GDS (sistemas de reserva usados por agências) | categórica (5) | na reserva | Repete o segmento? |
| `agent` | Código da agência | código (333) | na reserva | Centenas de códigos sem significado para o público. |
| `company` | Código da empresa que paga | código (352) | na reserva | 94% vazia: ajuda em algo? |
| `customer_type` | Transient, Transient-Party, Contract, Group | categórica (4) | na reserva | Cliente avulso cancela mais que contrato e grupo? |
| **Histórico do cliente** | | | | |
| `is_repeated_guest` | Já se hospedou antes | binária | na reserva | Quem volta cancela menos? |
| `previous_cancellations` | Reservas anteriores canceladas pelo cliente | contagem | na reserva | Quem já cancelou cancela de novo? |
| `previous_bookings_not_canceled` | Reservas anteriores que o cliente manteve | contagem | na reserva | Repete o `is_repeated_guest`? |
| **Quarto, pedidos e pagamento** | | | | |
| `reserved_room_type` | Tipo de quarto reservado (código anônimo) | categórica (10) | na reserva | O código não significa nada para o público. |
| `assigned_room_type` | Tipo de quarto entregue | categórica (12) | só no fim | Definido no check-in: quem cancelou nunca recebeu quarto. |
| `booking_changes` | Alterações até o check-in ou o cancelamento | contagem | só no fim | No momento da reserva, é sempre zero. |
| `deposit_type` | No Deposit, Non Refund (pago, sem reembolso), Refundable | categórica (3) | na reserva | Quem pagou sem reembolso cancela menos? A ação em discussão (pedir depósito) só vale para quem ainda não deu garantia. |
| `days_in_waiting_list` | Dias em lista de espera até a confirmação | numérica | na reserva | Esperar muito leva a desistir? |
| `adr` | Diária média | numérica | na reserva | Há diárias negativas ou zeradas? Diária alta, mais cancelamento? |
| `required_car_parking_spaces` | Vagas de garagem pedidas | contagem | ? | Pode ser pedida só na chegada. Se for, é vazamento. |
| `total_of_special_requests` | Pedidos especiais (cama de casal, andar alto...) | contagem | pode mudar até a chegada | Quem faz pedido pretende vir? |
| **Status da reserva** | | | | |
| `reservation_status` | Check-Out, Canceled, No-Show | categórica (3) | só no fim | É o próprio `y` escrito de outro jeito. |
| `reservation_status_date` | Data do último status | data | só no fim | Para quem cancelou, é a data do cancelamento. |
| **Variável resposta** | | | | |
| `is_canceled` | 1 = cancelada (inclui no-show), 0 = o hóspede veio | binária | | Qual a taxa de cancelamento? |